In [20]:
import pandas as pd

# read the 203_all_data.csv and 204_all_data.csv files
df_203 = pd.read_csv('Final_datasets/203_all_data.csv')
df_204 = pd.read_csv('Final_datasets/204_all_data.csv')

In [21]:
# add another column at 1st colimn position(0 indexed) named roomNo and put its value according to the last column. If last column value is 0, 
# then it is Outside, otherwisw 203 for df_203 and 204 for df_204
df_203.insert(0, 'roomNo', df_203.iloc[:, -1].apply(lambda x: 'Outside' if x == 0 else '203'))
df_204.insert(0, 'roomNo', df_204.iloc[:, -1].apply(lambda x: 'Outside' if x == 0 else '204'))

# drop the last column
df_203 = df_203.iloc[:, :-1]
df_204 = df_204.iloc[:, :-1]

# add a new column named entryId ats the start of both datasets and put its values as an index(incremental 1 indexed) for both datasets
df_203.insert(0, 'entryId', range(1, len(df_203) + 1))
df_204.insert(0, 'entryId', range(1, len(df_204) + 1))

# now rename the 2nd columns(0 indexed) of both datasets to entryDataPoint
df_203.rename(columns={df_203.columns[2]: 'entryDataPoint'}, inplace=True)
df_204.rename(columns={df_204.columns[2]: 'entryDataPoint'}, inplace=True)

# remove the entryDataPoint column from both datasets, then another 2 data with only entryId and entryDataPoint columns mapping
df_203_mapping = df_203[['entryId', 'entryDataPoint']].copy()
df_204_mapping = df_204[['entryId', 'entryDataPoint']].copy()
df_203.drop(columns=['entryDataPoint'], inplace=True)
df_204.drop(columns=['entryDataPoint'], inplace=True)

# save the modified datasets to new csv files
df_203.to_csv('203_matrix.csv', index=False)
df_204.to_csv('204_matrix.csv', index=False)
df_203_mapping.to_csv('203_entries.csv', index=False)
df_204_mapping.to_csv('204_entries.csv', index=False)

In [24]:
# Load the processed matrices and mappings
df_203 = pd.read_csv('203_matrix.csv')
df_204 = pd.read_csv('204_matrix.csv')
map_203 = pd.read_csv('203_entries.csv')
map_204 = pd.read_csv('204_entries.csv')

# Keep only the common AP columns
common_cols = list(set(df_203.columns) & set(df_204.columns))
common_cols = [col for col in common_cols if col not in ['entryId', 'roomNo']]

# Reduce both dataframes to these common columns
df_203_common = df_203[['entryId', 'roomNo'] + common_cols].copy()
df_204_common = df_204[['entryId', 'roomNo'] + common_cols].copy()

# Merge both datasets (stack one after another)
merged_df = pd.concat([df_203_common, df_204_common], ignore_index=True)

# Reassign new entryId (1-indexed)
if 'entryId' in merged_df.columns:
    merged_df.drop(columns=['entryId'], inplace=True)
merged_df.insert(0, 'entryId', range(1, len(merged_df) + 1))

# Merge mapping files
merged_mapping = pd.concat([map_203, map_204], ignore_index=True)
if 'entryId' in merged_mapping.columns:
    merged_mapping.drop(columns=['entryId'], inplace=True)
merged_mapping.insert(0, 'entryId', range(1, len(merged_mapping) + 1))

merged_df.to_csv('merged_matrix.csv', index=False)
merged_mapping.to_csv('merged_entries.csv', index=False)

print(f"Common AP columns retained: {len(common_cols)}")
print(f"Final merged dataset shape: {merged_df.shape}")


Common AP columns retained: 11
Final merged dataset shape: (127, 13)


In [2]:
# this code block takes 203_entries.csv, 204_entries.csv and merged_entries.csv files and checks the entryDataPoint column. Adds another column in that files, named entryType. If entryDataPoint contains the word "testing", then entryType should be "test", otherwise it should be "ref".

import pandas as pd
def classify_entry_type(file_path):
    df = pd.read_csv(file_path)
    df['entryType'] = df['entryDataPoint'].apply(lambda x: 'test' if 'testing' in x.lower() else 'ref')
    df.to_csv(file_path, index=False)
classify_entry_type('203_entries.csv')
classify_entry_type('204_entries.csv')
classify_entry_type('merged_entries.csv')